# 01: Regression Fundamentals — From Linear to Regularized

**Module 2, Week 5 — Machine Learning & Deep Learning**

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand the math** behind linear regression (cost function, gradient descent)
2. **Implement linear regression from scratch** using NumPy
3. **Use scikit-learn** to fit linear, polynomial, and regularized regression models
4. **Evaluate models** using MSE, RMSE, MAE, R², and Adjusted R²
5. **Apply regularization** (Ridge / Lasso / ElasticNet) to prevent overfitting
6. **Use cross-validation** to select hyperparameters and compare models
7. **Interpret feature importance** from standardized coefficients

---

> **Why regression?** It is the workhorse of predictive modeling. Nearly every ML practitioner starts here, and the concepts (cost functions, optimization, regularization, evaluation) carry directly into deep learning.

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
import sys
sys.path.insert(0, '../../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.datasets import make_regression

from shared.viz_utils import setup_style, save_fig
setup_style()

np.random.seed(42)
print("Setup complete.")

---

## Section 1: The Intuition Behind Linear Regression

### What is linear regression?

Linear regression finds the **best-fit line** (or hyperplane) through your data. The model assumes a linear relationship between input features and the target:

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_n x_n + \varepsilon$$

Where:
- $\hat{y}$ is the predicted value
- $\beta_0$ is the **intercept** (bias term)
- $\beta_1, \ldots, \beta_n$ are the **coefficients** (weights)
- $x_1, \ldots, x_n$ are the **features**
- $\varepsilon$ is the **error term** (noise we cannot explain)

### The Cost Function

"Best fit" means we want to **minimize the sum of squared residuals** — the difference between our predictions and the actual values:

$$J(\beta) = \frac{1}{2n} \sum_{i=1}^{n} (\hat{y}_i - y_i)^2$$

This is called **Mean Squared Error** (with a factor of 1/2 for mathematical convenience during differentiation).

### Why squared errors?
- Squaring **penalizes large errors more** than small ones
- The function is **smooth and differentiable** (so gradient descent works)
- It has a **single global minimum** (the function is convex)

### How do we minimize it?
Two approaches:
1. **Normal Equation** (closed-form): $\beta = (X^TX)^{-1}X^Ty$ — exact but expensive for large data
2. **Gradient Descent** (iterative): update weights step by step — scalable to millions of samples

### From-Scratch Implementation: Gradient Descent

Let's build linear regression from the ground up. We will:
1. Generate simple 1D data with a known relationship: $y = 3x + 7 + \text{noise}$
2. Implement gradient descent to learn the weight and bias
3. Watch the loss decrease over iterations

In [ ]:
# ── Generate simple 1D data ──────────────────────────────────────────────────
np.random.seed(42)
n_samples = 100
X_simple = 2 * np.random.rand(n_samples, 1)          # feature in [0, 2]
y_simple = 3 * X_simple.flatten() + 7 + np.random.randn(n_samples) * 0.8  # y = 3x + 7 + noise

print(f"True relationship: y = 3x + 7")
print(f"Data shape: X={X_simple.shape}, y={y_simple.shape}")
print(f"First 5 X values: {X_simple[:5].flatten()}")
print(f"First 5 y values: {y_simple[:5].round(2)}")

In [ ]:
# ── Gradient Descent from Scratch ────────────────────────────────────────────

def linear_regression_gd(X, y, learning_rate=0.1, n_iterations=200):
    """
    Train a simple linear regression using gradient descent.
    
    Parameters
    ----------
    X : array of shape (n_samples, 1)
    y : array of shape (n_samples,)
    learning_rate : step size for weight updates
    n_iterations : number of gradient descent steps
    
    Returns
    -------
    weight, bias, loss_history
    """
    n = len(y)
    
    # Step 1: Initialize weights randomly (small values)
    weight = np.random.randn() * 0.01
    bias = 0.0
    loss_history = []
    
    for i in range(n_iterations):
        # Step 2: Forward pass — compute predictions
        y_pred = weight * X.flatten() + bias
        
        # Step 3: Compute the loss (MSE)
        loss = np.mean((y_pred - y) ** 2) / 2
        loss_history.append(loss)
        
        # Step 4: Compute gradients
        #   dJ/d(weight) = (1/n) * sum((y_pred - y) * X)
        #   dJ/d(bias)   = (1/n) * sum(y_pred - y)
        error = y_pred - y
        grad_weight = np.mean(error * X.flatten())
        grad_bias = np.mean(error)
        
        # Step 5: Update weights (move opposite to gradient)
        weight -= learning_rate * grad_weight
        bias -= learning_rate * grad_bias
        
        # Print progress every 50 iterations
        if (i + 1) % 50 == 0:
            print(f"  Iteration {i+1:>4d} | Loss: {loss:.4f} | "
                  f"weight: {weight:.4f} | bias: {bias:.4f}")
    
    return weight, bias, loss_history

print("Training linear regression with gradient descent...\n")
learned_weight, learned_bias, losses = linear_regression_gd(X_simple, y_simple)

print(f"\n--- Results ---")
print(f"Learned:  y = {learned_weight:.4f}x + {learned_bias:.4f}")
print(f"Actual:   y = 3.0000x + 7.0000")

In [ ]:
# ── Visualize: Data + Fitted Line, and Loss Curve ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: scatter + fitted line
axes[0].scatter(X_simple, y_simple, alpha=0.6, s=30, label='Data points')
x_line = np.linspace(0, 2, 100)
y_line = learned_weight * x_line + learned_bias
axes[0].plot(x_line, y_line, 'r-', linewidth=2,
             label=f'Fit: y = {learned_weight:.2f}x + {learned_bias:.2f}')
axes[0].plot(x_line, 3 * x_line + 7, 'g--', linewidth=1.5, alpha=0.7,
             label='True: y = 3x + 7')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].set_title('Linear Regression — From Scratch')
axes[0].legend()

# Right: loss curve
axes[1].plot(losses, linewidth=2, color='darkorange')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Loss (MSE / 2)')
axes[1].set_title('Training Loss Over Time')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()
print("The loss drops quickly at first, then plateaus as we approach the optimum.")

### Sanity Check: sklearn's LinearRegression

Let's verify our from-scratch implementation gives the same answer as scikit-learn.

In [ ]:
# ── sklearn Linear Regression on the same data ──────────────────────────────
lr = LinearRegression()
lr.fit(X_simple, y_simple)

print("sklearn LinearRegression (uses Normal Equation internally):")
print(f"  Coefficient (slope): {lr.coef_[0]:.4f}")
print(f"  Intercept (bias):    {lr.intercept_:.4f}")
print()
print("Our gradient descent result:")
print(f"  Coefficient (slope): {learned_weight:.4f}")
print(f"  Intercept (bias):    {learned_bias:.4f}")
print()
print("True values:")
print(f"  Coefficient (slope): 3.0000")
print(f"  Intercept (bias):    7.0000")
print()
print("They match closely! Small differences come from:")
print("  - Gradient descent: finite iterations (approximate)")
print("  - Normal equation: exact closed-form solution")
print("  - Both differ from true values because of noise in data")

---

## Section 2: Multiple Regression

In reality, we rarely have just one feature. **Multiple regression** extends the model to handle many input features simultaneously:

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_n x_n$$

Each coefficient $\beta_j$ tells us: *"Holding all other features constant, a one-unit increase in $x_j$ changes $\hat{y}$ by $\beta_j$ units."*

Let's create a synthetic housing dataset with realistic features and see how multiple regression handles it.

In [ ]:
# ── Create a synthetic housing dataset ────────────────────────────────────────
np.random.seed(42)
n = 1000

housing = pd.DataFrame({
    'sqft':             np.random.normal(1800, 400, n).clip(600, 4000),
    'bedrooms':         np.random.choice([1, 2, 3, 4, 5], n, p=[0.05, 0.2, 0.4, 0.25, 0.1]),
    'age_years':        np.random.exponential(15, n).clip(0, 80).astype(int),
    'distance_to_city': np.random.exponential(10, n).clip(1, 50),
    'has_garage':       np.random.choice([0, 1], n, p=[0.3, 0.7]),
})

# True relationship (with some noise)
housing['price'] = (
    150 * housing['sqft']
    + 20000 * housing['bedrooms']
    - 1500 * housing['age_years']
    - 3000 * housing['distance_to_city']
    + 25000 * housing['has_garage']
    + 50000                                  # base price
    + np.random.normal(0, 20000, n)          # noise
)

print("Synthetic Housing Dataset")
print(f"Shape: {housing.shape}")
print()
housing.head(8)

In [ ]:
# ── Fit Multiple Regression ───────────────────────────────────────────────────
feature_cols = ['sqft', 'bedrooms', 'age_years', 'distance_to_city', 'has_garage']
X_housing = housing[feature_cols]
y_housing = housing['price']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_housing, y_housing, test_size=0.2, random_state=42
)

# Fit the model
lr_multi = LinearRegression()
lr_multi.fit(X_train, y_train)

# Display coefficients
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Learned Coefficient': lr_multi.coef_.round(2),
    'True Coefficient': [150, 20000, -1500, -3000, 25000]
})
print("Multiple Regression Coefficients:")
print(f"  Intercept: {lr_multi.intercept_:.2f} (true: 50000)")
print()
print(coef_df.to_string(index=False))

In [ ]:
# ── Coefficient bar chart ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
sorted_idx = np.argsort(np.abs(lr_multi.coef_))[::-1]
colors = ['#2ecc71' if c > 0 else '#e74c3c' for c in lr_multi.coef_[sorted_idx]]

bars = ax.barh(
    [feature_cols[i] for i in sorted_idx],
    lr_multi.coef_[sorted_idx],
    color=colors, edgecolor='white', linewidth=0.5
)
ax.set_xlabel('Coefficient Value')
ax.set_title('Multiple Regression — Feature Coefficients')
ax.axvline(x=0, color='gray', linestyle='--', linewidth=0.8)

# Add value labels
for bar, val in zip(bars, lr_multi.coef_[sorted_idx]):
    ax.text(val + (500 if val > 0 else -500), bar.get_y() + bar.get_height()/2,
            f'{val:,.0f}', va='center', ha='left' if val > 0 else 'right', fontsize=10)

plt.tight_layout()
plt.show()
print("Green = positive effect on price, Red = negative effect")

---

## Section 3: Evaluation Metrics

How do we know if our regression model is any good? We need **metrics** to quantify performance.

### The Key Metrics

| Metric | Formula | Interpretation |
|--------|---------|---------------|
| **MSE** (Mean Squared Error) | $\frac{1}{n}\sum(\hat{y}_i - y_i)^2$ | Average squared error. Penalizes large errors heavily. Units are squared. |
| **RMSE** (Root MSE) | $\sqrt{MSE}$ | Same as MSE but in original units. Easier to interpret. |
| **MAE** (Mean Absolute Error) | $\frac{1}{n}\sum\|\hat{y}_i - y_i\|$ | Average absolute error. More robust to outliers than MSE. |
| **R²** (Coefficient of Determination) | $1 - \frac{SS_{res}}{SS_{tot}}$ | Proportion of variance explained. 1.0 = perfect, 0.0 = as good as predicting the mean. |
| **Adjusted R²** | $1 - \frac{(1-R^2)(n-1)}{n-p-1}$ | R² penalized for number of features. Prevents rewarding unnecessary features. |

### When to use each?
- **RMSE** is the default go-to metric (interpretable units, penalizes big errors)
- **MAE** when you want robustness to outliers
- **R²** for a quick "how good is this?" summary (but beware: it always increases with more features)
- **Adjusted R²** when comparing models with different numbers of features

In [ ]:
# ── Compute all evaluation metrics ────────────────────────────────────────────
y_pred_housing = lr_multi.predict(X_test)

mse  = mean_squared_error(y_test, y_pred_housing)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred_housing)
r2   = r2_score(y_test, y_pred_housing)

# Adjusted R²
n_test = len(y_test)
p = X_test.shape[1]
adj_r2 = 1 - (1 - r2) * (n_test - 1) / (n_test - p - 1)

metrics_df = pd.DataFrame({
    'Metric': ['MSE', 'RMSE', 'MAE', 'R²', 'Adjusted R²'],
    'Value': [f'{mse:,.0f}', f'{rmse:,.0f}', f'{mae:,.0f}', f'{r2:.4f}', f'{adj_r2:.4f}'],
    'Interpretation': [
        f'Average squared error (in dollars²)',
        f'Typical prediction is off by ~${rmse:,.0f}',
        f'Average absolute error of ~${mae:,.0f}',
        f'{r2*100:.1f}% of variance explained',
        f'R² adjusted for {p} features'
    ]
})

print("Model Evaluation on Test Set:")
print("=" * 70)
print(metrics_df.to_string(index=False))
print("=" * 70)

In [ ]:
# ── Diagnostic Plots: Actual vs Predicted + Residuals ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Actual vs Predicted
axes[0].scatter(y_test, y_pred_housing, alpha=0.4, s=20)
min_val = min(y_test.min(), y_pred_housing.min())
max_val = max(y_test.max(), y_pred_housing.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2,
             label='Perfect predictions')
axes[0].set_xlabel('Actual Price ($)')
axes[0].set_ylabel('Predicted Price ($)')
axes[0].set_title(f'Actual vs Predicted (R² = {r2:.3f})')
axes[0].legend()

# Right: Residuals distribution
residuals = y_test - y_pred_housing
axes[1].hist(residuals, bins=40, edgecolor='white', alpha=0.7, color='steelblue')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residuals Distribution')
axes[1].annotate(f'Mean: ${residuals.mean():,.0f}\nStd: ${residuals.std():,.0f}',
                 xy=(0.72, 0.85), xycoords='axes fraction', fontsize=10,
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()
print("A good model has: points clustered around the red line (left),")
print("and residuals centered at zero with a bell shape (right).")

---

## Section 4: Polynomial Regression

### What if the relationship is NOT linear?

Sometimes the true relationship between features and target is **curved**. A straight line will never capture it well. Polynomial regression adds powers of the features:

- Degree 1 (linear): $\hat{y} = \beta_0 + \beta_1 x$
- Degree 2 (quadratic): $\hat{y} = \beta_0 + \beta_1 x + \beta_2 x^2$
- Degree 3 (cubic): $\hat{y} = \beta_0 + \beta_1 x + \beta_2 x^2 + \beta_3 x^3$

sklearn's `PolynomialFeatures` creates these extra columns for us.

> **Warning:** High-degree polynomials can **overfit** — they memorize the training data (including noise) and perform terribly on new data. We will see this clearly below.

In [ ]:
# ── Generate nonlinear data ───────────────────────────────────────────────────
np.random.seed(42)
n_poly = 80

X_poly = np.sort(np.random.uniform(-3, 3, n_poly)).reshape(-1, 1)
y_poly = 0.5 * X_poly.flatten()**2 - 2 * X_poly.flatten() + 3 + np.random.randn(n_poly) * 1.5

# Train/test split
X_poly_train, X_poly_test, y_poly_train, y_poly_test = train_test_split(
    X_poly, y_poly, test_size=0.25, random_state=42
)

print(f"True function: y = 0.5x² - 2x + 3 + noise")
print(f"Training samples: {len(X_poly_train)}, Test samples: {len(X_poly_test)}")

In [ ]:
# ── Fit polynomials of different degrees and compare ─────────────────────────
degrees = [1, 2, 5, 15]
x_plot = np.linspace(-3, 3, 300).reshape(-1, 1)

fig, ax = plt.subplots(figsize=(12, 6))
ax.scatter(X_poly_train, y_poly_train, alpha=0.6, s=40, label='Train data', zorder=5)
ax.scatter(X_poly_test, y_poly_test, alpha=0.6, s=40, marker='x',
           label='Test data', zorder=5, color='red')

results = []
colors_deg = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c']

for deg, color in zip(degrees, colors_deg):
    # Create a pipeline: polynomial features -> linear regression
    model = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('lr', LinearRegression())
    ])
    model.fit(X_poly_train, y_poly_train)
    
    # Scores
    train_r2 = model.score(X_poly_train, y_poly_train)
    test_r2 = model.score(X_poly_test, y_poly_test)
    
    # Plot the fit
    y_plot = model.predict(x_plot)
    ax.plot(x_plot, y_plot, color=color, linewidth=2,
            label=f'Degree {deg} (test R²={test_r2:.3f})')
    
    results.append({
        'Degree': deg,
        'Num Features': PolynomialFeatures(degree=deg, include_bias=False).fit_transform(X_poly_train).shape[1],
        'Train R²': f'{train_r2:.4f}',
        'Test R²': f'{test_r2:.4f}',
        'Overfit?': 'Yes' if train_r2 - test_r2 > 0.1 else 'No'
    })

ax.set_ylim(-5, 20)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Polynomial Regression — Underfitting vs Overfitting')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

# Summary table
print("\nPolynomial Degree Comparison:")
print("=" * 65)
print(pd.DataFrame(results).to_string(index=False))
print("=" * 65)
print("\nNotice: Degree 2 matches the true function and generalizes well.")
print("Degree 15 has near-perfect train R² but terrible test R² — that is overfitting!")

---

## Section 5: Regularization — Taming Overfitting

### The Problem

When a model has too many features or too much flexibility (e.g., high-degree polynomial), it can **memorize** the training data — including its noise. The model fits the training set perfectly but fails on new data.

### The Solution: Add a Penalty

Regularization modifies the cost function by adding a **penalty term** that discourages large coefficients:

$$J(\beta) = \frac{1}{2n} \sum(\hat{y}_i - y_i)^2 + \alpha \cdot \text{penalty}(\beta)$$

The hyperparameter $\alpha$ controls the trade-off between fitting the data and keeping coefficients small.

### Three Flavors of Regularization

| Method | Penalty Term | Effect on Coefficients | Best When |
|--------|-------------|----------------------|-----------|
| **Ridge** (L2) | $\alpha \sum \beta_j^2$ | Shrinks all coefficients toward zero, but never exactly zero | Many features are all somewhat useful |
| **Lasso** (L1) | $\alpha \sum \|\beta_j\|$ | Drives some coefficients to **exactly zero** (automatic feature selection) | You suspect only a few features matter |
| **ElasticNet** (L1 + L2) | $\alpha_1 \sum \|\beta_j\| + \alpha_2 \sum \beta_j^2$ | Combines both effects | Groups of correlated features |

### Key Intuition
- **Small $\alpha$**: Minimal regularization, behaves like ordinary regression
- **Large $\alpha$**: Heavy regularization, model becomes too simple (underfitting)
- **Just right $\alpha$**: Best generalization — we find it via cross-validation

In [ ]:
# ── Regularization rescues the degree-15 polynomial ──────────────────────────

# Build degree-15 polynomial features
poly15 = PolynomialFeatures(degree=15, include_bias=False)
X_poly15_train = poly15.fit_transform(X_poly_train)
X_poly15_test = poly15.transform(X_poly_test)
X_poly15_plot = poly15.transform(x_plot)

# Compare: plain OLS vs Ridge vs Lasso at degree 15
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models_reg = {
    'Linear (no regularization)': LinearRegression(),
    'Ridge (alpha=1.0)': Ridge(alpha=1.0),
    'Lasso (alpha=0.1)': Lasso(alpha=0.1, max_iter=10000),
}

for ax, (name, model) in zip(axes, models_reg.items()):
    model.fit(X_poly15_train, y_poly_train)
    train_r2 = model.score(X_poly15_train, y_poly_train)
    test_r2 = model.score(X_poly15_test, y_poly_test)
    
    ax.scatter(X_poly_train, y_poly_train, alpha=0.5, s=30, label='Train')
    ax.scatter(X_poly_test, y_poly_test, alpha=0.5, s=30, marker='x', label='Test')
    
    y_plot_reg = model.predict(X_poly15_plot)
    ax.plot(x_plot, y_plot_reg, 'r-', linewidth=2)
    
    ax.set_ylim(-5, 20)
    ax.set_title(f'{name}\nTrain R²={train_r2:.3f}, Test R²={test_r2:.3f}')
    ax.set_xlabel('x')
    ax.legend(fontsize=8)

axes[0].set_ylabel('y')
plt.suptitle('Degree-15 Polynomial: Regularization to the Rescue', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Without regularization, degree 15 wildly overfits.")
print("Ridge and Lasso constrain the coefficients, producing smooth, sensible curves.")

In [ ]:
# ── Regularization Path: Coefficients vs Alpha ───────────────────────────────
alphas = np.logspace(-4, 4, 100)

ridge_coefs = []
lasso_coefs = []

for alpha in alphas:
    # Ridge
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_poly15_train, y_poly_train)
    ridge_coefs.append(ridge.coef_)
    
    # Lasso
    lasso = Lasso(alpha=alpha, max_iter=10000)
    lasso.fit(X_poly15_train, y_poly_train)
    lasso_coefs.append(lasso.coef_)

ridge_coefs = np.array(ridge_coefs)
lasso_coefs = np.array(lasso_coefs)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Ridge path
for i in range(ridge_coefs.shape[1]):
    axes[0].plot(alphas, ridge_coefs[:, i], linewidth=0.8)
axes[0].set_xscale('log')
axes[0].set_xlabel('Alpha (regularization strength)')
axes[0].set_ylabel('Coefficient value')
axes[0].set_title('Ridge — Coefficients shrink smoothly toward zero')
axes[0].axhline(y=0, color='black', linestyle='--', linewidth=0.5)

# Lasso path
for i in range(lasso_coefs.shape[1]):
    axes[1].plot(alphas, lasso_coefs[:, i], linewidth=0.8)
axes[1].set_xscale('log')
axes[1].set_xlabel('Alpha (regularization strength)')
axes[1].set_ylabel('Coefficient value')
axes[1].set_title('Lasso — Coefficients are driven to exactly zero')
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=0.5)

plt.tight_layout()
plt.show()

print("Key difference:")
print("  Ridge: All coefficients shrink but stay non-zero (no feature is fully removed)")
print("  Lasso: Many coefficients hit exactly zero (automatic feature selection!)")

---

## Section 6: Cross-Validation

### Why a single train/test split is not enough

A single train/test split can be **lucky or unlucky**. The test score depends heavily on which samples landed in which set. We need a more robust way to estimate model performance.

### K-Fold Cross-Validation

1. Split the data into **K equal folds** (typically K=5 or K=10)
2. For each fold:
   - Train on K-1 folds
   - Evaluate on the held-out fold
3. Average the K scores to get a **more reliable estimate**

```
Fold 1: [TEST] [train] [train] [train] [train]
Fold 2: [train] [TEST] [train] [train] [train]
Fold 3: [train] [train] [TEST] [train] [train]
Fold 4: [train] [train] [train] [TEST] [train]
Fold 5: [train] [train] [train] [train] [TEST]
```

**Benefits:**
- Every sample gets to be in the test set exactly once
- We get a mean AND standard deviation of the score (measures reliability)
- Much better estimate of how the model will perform on truly new data

In [ ]:
# ── Compare models with 5-fold Cross-Validation on housing data ──────────────

# Scale features for regularized models (important!)
scaler = StandardScaler()
X_housing_scaled = scaler.fit_transform(X_housing)

models_cv = {
    'Linear Regression': LinearRegression(),
    'Ridge (alpha=1)': Ridge(alpha=1.0),
    'Ridge (alpha=10)': Ridge(alpha=10.0),
    'Lasso (alpha=1)': Lasso(alpha=1.0, max_iter=10000),
    'Lasso (alpha=10)': Lasso(alpha=10.0, max_iter=10000),
    'ElasticNet (alpha=1)': ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000),
}

cv_results = {}
print("5-Fold Cross-Validation Results (R² score):")
print("=" * 60)

for name, model in models_cv.items():
    scores = cross_val_score(model, X_housing_scaled, y_housing, 
                             cv=5, scoring='r2')
    cv_results[name] = scores
    print(f"  {name:<25s}  mean={scores.mean():.4f}  std={scores.std():.4f}")

print("=" * 60)

In [ ]:
# ── Box plot comparison ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))

cv_df = pd.DataFrame(cv_results)
cv_melted = cv_df.melt(var_name='Model', value_name='R² Score')

sns.boxplot(data=cv_melted, x='Model', y='R² Score', ax=ax, palette='Set2')
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
ax.set_title('5-Fold Cross-Validation: Model Comparison')
ax.set_ylabel('R² Score')
ax.set_xlabel('')

plt.tight_layout()
plt.show()

print("Box plots show the spread of R² across folds.")
print("Models with tight boxes are more consistent/reliable.")

---

## Section 7: Feature Importance via Standardized Coefficients

### The problem with raw coefficients

Raw coefficients are **not directly comparable** because features have different scales:
- `sqft` ranges from 600 to 4000 → its coefficient is ~150
- `has_garage` is 0 or 1 → its coefficient is ~25000

This does NOT mean `has_garage` is more important! To compare fairly, we need to **standardize** the features first (zero mean, unit variance), then look at the coefficients.

A **standardized coefficient** tells us: *"How much does the target change (in standard deviations) when this feature increases by one standard deviation?"*

In [ ]:
# ── Feature Importance: Standardized Coefficients ────────────────────────────

# Fit on scaled data
lr_scaled = LinearRegression()
lr_scaled.fit(X_housing_scaled, y_housing)

# Create importance dataframe
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Raw Coefficient': lr_multi.coef_.round(1),
    'Standardized Coefficient': lr_scaled.coef_.round(1),
    'Abs Standardized': np.abs(lr_scaled.coef_).round(1),
}).sort_values('Abs Standardized', ascending=True)

print("Feature Importance (Standardized Coefficients):")
print(importance_df[['Feature', 'Raw Coefficient', 'Standardized Coefficient']].to_string(index=False))

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
colors_imp = ['#2ecc71' if c > 0 else '#e74c3c' for c in importance_df['Standardized Coefficient']]

bars = ax.barh(importance_df['Feature'], importance_df['Standardized Coefficient'],
               color=colors_imp, edgecolor='white')
ax.set_xlabel('Standardized Coefficient')
ax.set_title('Feature Importance — After Standardization')
ax.axvline(x=0, color='gray', linestyle='--', linewidth=0.8)

for bar, val in zip(bars, importance_df['Standardized Coefficient']):
    offset = 500 if val > 0 else -500
    ax.text(val + offset, bar.get_y() + bar.get_height()/2,
            f'{val:,.0f}', va='center', ha='left' if val > 0 else 'right', fontsize=10)

plt.tight_layout()
plt.show()

print("\nNow we can fairly compare: the feature with the largest absolute")
print("standardized coefficient has the strongest influence on price.")

---

## Key Takeaways

### Core Concepts
- **Linear regression** minimizes the sum of squared residuals to find the best-fit line/hyperplane
- **Gradient descent** iteratively updates weights by moving opposite to the gradient of the cost function
- **Multiple regression** extends the model to handle many features — each coefficient represents the effect of one feature, holding others constant

### Evaluation
- **RMSE** is the go-to metric (same units as the target, penalizes big errors)
- **R²** tells you what proportion of variance your model explains (1.0 = perfect)
- Always check **residual plots** — they reveal patterns your model missed

### Regularization
- **Ridge (L2)**: Shrinks all coefficients — use when most features are useful
- **Lasso (L1)**: Zeros out unimportant coefficients — use for feature selection
- **ElasticNet**: Best of both worlds — use with correlated features

### Best Practices
- **Always scale features** before applying regularization
- **Use cross-validation** (not a single train/test split) to choose hyperparameters
- **Standardize coefficients** before comparing feature importance

### Decision Flowchart

```
Start with Linear Regression
    │
    ├── Check residual plots
    │       │
    │       ├── Pattern visible? → Try Polynomial Features
    │       └── Random scatter? → Good, linear is appropriate
    │
    ├── Train R² >> Test R²? (Overfitting?)
    │       │
    │       ├── Yes → Apply regularization (Ridge/Lasso)
    │       └── No  → Model is generalizing well
    │
    └── Use Cross-Validation to pick the best alpha
```

---

## Exercises

Practice what you have learned. Each exercise builds on the concepts from this notebook.

### Exercise 1: Ridge Regression on the Diabetes Dataset

Load sklearn's built-in `diabetes` dataset. Build a Ridge regression and find the optimal `alpha` using cross-validation.

**Steps:**
1. Load the data with `from sklearn.datasets import load_diabetes`
2. Scale the features with `StandardScaler`
3. Try alpha values: `[0.001, 0.01, 0.1, 1, 10, 100, 1000]`
4. Use `cross_val_score` with 5-fold CV and `scoring='r2'` for each alpha
5. Plot mean R² vs alpha (use log scale for x-axis)
6. Print the best alpha and its R² score

In [ ]:
# ── Exercise 1: Your code here ────────────────────────────────────────────────
from sklearn.datasets import load_diabetes

# Load data
diabetes = load_diabetes()
X_diab, y_diab = diabetes.data, diabetes.target

# TODO: Scale the features
# scaler = ...

# TODO: Try different alphas and compute CV scores
# alphas_to_try = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
# mean_scores = []
# for alpha in alphas_to_try:
#     ...

# TODO: Plot mean R² vs alpha

# TODO: Print the best alpha

### Exercise 2: Lasso Feature Selection

Use `make_regression` to create a dataset with 20 features but only 5 are truly informative. Use Lasso to automatically discover which features matter.

**Steps:**
1. Generate data: `make_regression(n_samples=500, n_features=20, n_informative=5, noise=10, random_state=42)`
2. Scale the features
3. Fit Lasso with several alpha values (e.g., 0.01, 0.1, 1, 10)
4. For each alpha, print how many coefficients are non-zero
5. For the best alpha, show a bar chart of the coefficients — verify that roughly 5 features survive

In [ ]:
# ── Exercise 2: Your code here ────────────────────────────────────────────────

# TODO: Generate data with make_regression
# X_ex2, y_ex2 = make_regression(n_samples=500, n_features=20, n_informative=5,
#                                 noise=10, random_state=42)

# TODO: Scale features

# TODO: Fit Lasso with different alphas and count non-zero coefficients
# for alpha in [0.01, 0.1, 1, 10]:
#     lasso = Lasso(alpha=alpha, max_iter=10000)
#     lasso.fit(X_ex2_scaled, y_ex2)
#     n_nonzero = np.sum(lasso.coef_ != 0)
#     print(f"alpha={alpha:>5.2f}  non-zero coefficients: {n_nonzero}/20")

# TODO: Bar chart of coefficients for the best alpha

### Exercise 3: Full Pipeline — Scale, Polynomial, Ridge

Build a complete sklearn `Pipeline` that chains: `StandardScaler` → `PolynomialFeatures(degree=2)` → `Ridge`.

Evaluate it with 5-fold cross-validation on the housing dataset.

**Steps:**
1. Create a pipeline with three steps: `('scaler', StandardScaler())`, `('poly', PolynomialFeatures(degree=2))`, `('ridge', Ridge(alpha=1.0))`
2. Use `cross_val_score` with `cv=5` and `scoring='r2'`
3. Compare against a plain `LinearRegression` pipeline (scaler + LR)
4. Print both results and determine if polynomial features helped

In [ ]:
# ── Exercise 3: Your code here ────────────────────────────────────────────────

# TODO: Build the polynomial Ridge pipeline
# pipe_poly_ridge = Pipeline([
#     ('scaler', StandardScaler()),
#     ('poly', PolynomialFeatures(degree=2)),
#     ('ridge', Ridge(alpha=1.0))
# ])

# TODO: Build a simple Linear Regression pipeline for comparison
# pipe_lr = Pipeline([
#     ('scaler', StandardScaler()),
#     ('lr', LinearRegression())
# ])

# TODO: Evaluate both with cross_val_score on X_housing, y_housing
# scores_poly_ridge = cross_val_score(pipe_poly_ridge, X_housing, y_housing, cv=5, scoring='r2')
# scores_lr = cross_val_score(pipe_lr, X_housing, y_housing, cv=5, scoring='r2')

# TODO: Print and compare results
# print(f"Linear Regression:       R² = {scores_lr.mean():.4f} +/- {scores_lr.std():.4f}")
# print(f"Poly(2) + Ridge:         R² = {scores_poly_ridge.mean():.4f} +/- {scores_poly_ridge.std():.4f}")

---

**End of Notebook 01: Regression Fundamentals**

Next up: **02 — Classification** where we move from predicting continuous values to predicting categories.